In [1]:
import torch
import numpy as np

vocab = 6
dim = 3 

word1 = [0.31897191, 0.70165633, 0.87015771]
word2 = [0.66534041, 0.17288046, 0.60692252]
word3 = [0.05295369, 0.36576202, 0.38331766]
word4 = [0.30033345, 0.98547427, 0.18919171]
word5 = [0.66448752, 0.07724835, 0.56031583]
word6 = [0.75046948, 0.03739152, 0.90530486]

inputs = torch.tensor((word1, word2, word3, word4, word5, word6 ))
inputs

tensor([[0.3190, 0.7017, 0.8702],
        [0.6653, 0.1729, 0.6069],
        [0.0530, 0.3658, 0.3833],
        [0.3003, 0.9855, 0.1892],
        [0.6645, 0.0772, 0.5603],
        [0.7505, 0.0374, 0.9053]])

calculating the context vector of word 2 which involves calculating attention scores between word 2 and all other input embeddings through dot product

In [2]:
quary = inputs[1]
attn_score = (quary * inputs).sum(dim=1)
attn_score

tensor([0.8616, 0.8409, 0.3311, 0.4850, 0.7955, 1.0552])

So till now we have calculated attn_score which basically is 

word[1] dotp word[2]
word[2] dotp word[2]
word[3] dotp 
......

giving us a vector of 6 values 


In [3]:
attn_weight = attn_score/attn_score.sum()
attn_weight # sums to 1

tensor([0.1972, 0.1925, 0.0758, 0.1110, 0.1821, 0.2415])

in the step below we are basically doing step by step

defining an empty array,

context vect of 2nd word is calculated by submission of weights[i] * inputs[i]

context_vect += weights[i] * weights[i] // 1,3 + 1,3 + 1,3......6 times and collapsing to give 1,3

In [4]:
query = inputs[1]

context_vect = torch.zeros(query.shape) # [0., 0., 0.] 
context_vect_2 = context_vect

for i, x_i in enumerate(inputs):
    context_vect +=  attn_weight[i] * x_i

for i in range(6):
    context_vect_2 += attn_weight[i] * inputs[i]
    print(attn_weight[i], inputs[i])

context_vect_2

tensor(0.1972) tensor([0.3190, 0.7017, 0.8702])
tensor(0.1925) tensor([0.6653, 0.1729, 0.6069])
tensor(0.0758) tensor([0.0530, 0.3658, 0.3833])
tensor(0.1110) tensor([0.3003, 0.9855, 0.1892])
tensor(0.1821) tensor([0.6645, 0.0772, 0.5603])
tensor(0.2415) tensor([0.7505, 0.0374, 0.9053])


tensor([1.0610, 0.6637, 1.3182])

Computing the attention weihgts for all input tokens

In [5]:
values = inputs
atten_score = torch.zeros(values.shape)
atten_score.shape

torch.Size([6, 3])

every word is a 1,3
and }we have 6 words

so all 6 words will produces attn score of all the words, which gives us 6, 6 matrix, where first row represent the attn score of 1st word to the 
other words

In [6]:
attn_score = torch.zeros(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_score[i, j] = x_i.dot(x_j)

attn_score 
        

tensor([[1.3512, 0.8616, 0.6071, 0.9519, 0.7537, 1.0534],
        [0.8616, 0.8409, 0.3311, 0.4850, 0.7955, 1.0552],
        [0.6071, 0.3311, 0.2835, 0.4489, 0.2782, 0.4004],
        [0.9519, 0.4850, 0.4489, 1.0972, 0.3817, 0.4335],
        [0.7537, 0.7955, 0.2782, 0.3817, 0.7615, 1.0088],
        [1.0534, 1.0552, 0.4004, 0.4335, 1.0088, 1.3842]])

In [7]:
attn_score = inputs @ inputs.T
attn_score

tensor([[1.3512, 0.8616, 0.6071, 0.9519, 0.7537, 1.0534],
        [0.8616, 0.8409, 0.3311, 0.4850, 0.7955, 1.0552],
        [0.6071, 0.3311, 0.2835, 0.4489, 0.2782, 0.4004],
        [0.9519, 0.4850, 0.4489, 1.0972, 0.3817, 0.4335],
        [0.7537, 0.7955, 0.2782, 0.3817, 0.7615, 1.0088],
        [1.0534, 1.0552, 0.4004, 0.4335, 1.0088, 1.3842]])

In [8]:
attn_weights = attn_score.softmax(dim=-1)

In [9]:
attn_weights.shape

torch.Size([6, 6])

In [10]:
context_vect =  attn_weights @ inputs
context_vect

tensor([[0.4631, 0.4227, 0.6249],
        [0.5072, 0.3479, 0.6372],
        [0.4533, 0.4158, 0.5975],
        [0.4260, 0.4917, 0.5621],
        [0.5115, 0.3388, 0.6378],
        [0.5278, 0.3182, 0.6611]])

## Training self attention with Trainable weights

In [11]:
x_2 = inputs[1] # [0.6653, 0.1729, 0.6069]
d_in = inputs.shape[1] # 3
d_out =2 
d_in, d_out

(3, 2)

In [12]:
torch.manual_seed(123)

W_q = torch.nn.Linear(d_in, d_out)
W_k = torch.nn.Linear(d_in, d_out)
W_v = torch.nn.Linear(d_in, d_out)


In [18]:
q_2 = W_q(x_2)
k_2 = W_k(x_2)
v_2 = W_v(x_2)

In [20]:
q_2

tensor([-0.7470, -0.1423], grad_fn=<ViewBackward0>)

In [29]:
keys =  W_k(inputs)
keys.shape

torch.Size([6, 2])

In [31]:
query  = W_q(inputs)
query.shape

torch.Size([6, 2])

In [32]:
value = W_v(inputs)
value.shape

torch.Size([6, 2])

#### the unscalled attn score is calculated by the dot product of keys and query

In [38]:
keys_2 = keys[2]
attn_score = query[2].dot(keys_2)
attn_score

tensor(0.3096, grad_fn=<DotBackward0>)

In [41]:
attn_score = query[2] @ keys.T
attn_score

tensor([0.4993, 0.4126, 0.3096, 0.3338, 0.3912, 0.4914],
       grad_fn=<SqueezeBackward4>)

In [42]:
query[2].shape, keys.T.shape

(torch.Size([2]), torch.Size([2, 6]))

In [49]:
torch.softmax(attn_score/d_k**0.5, dim=-1)

tensor([0.1778, 0.1672, 0.1555, 0.1581, 0.1647, 0.1768],
       grad_fn=<SoftmaxBackward0>)

In [50]:
d_k = keys.shape[1]
attn_weight_2 = torch.softmax(attn_score/d_k**0.5, dim=-1)
attn_weight_2

tensor([0.1778, 0.1672, 0.1555, 0.1581, 0.1647, 0.1768],
       grad_fn=<SoftmaxBackward0>)

In [56]:
attn_weight.shape, values.shape

(torch.Size([6]), torch.Size([6, 3]))

In [61]:
context_vect = attn_weight_2 @ values
context_vect

tensor([0.4658, 0.3857, 0.5980], grad_fn=<SqueezeBackward4>)

In [64]:
attn_weight_2, values

(tensor([0.1778, 0.1672, 0.1555, 0.1581, 0.1647, 0.1768],
        grad_fn=<SoftmaxBackward0>),
 tensor([[0.3190, 0.7017, 0.8702],
         [0.6653, 0.1729, 0.6069],
         [0.0530, 0.3658, 0.3833],
         [0.3003, 0.9855, 0.1892],
         [0.6645, 0.0772, 0.5603],
         [0.7505, 0.0374, 0.9053]]))

In [120]:
d_in , d_out = 3, 2

In [121]:
inputs

tensor([[0.3190, 0.7017, 0.8702],
        [0.6653, 0.1729, 0.6069],
        [0.0530, 0.3658, 0.3833],
        [0.3003, 0.9855, 0.1892],
        [0.6645, 0.0772, 0.5603],
        [0.7505, 0.0374, 0.9053]])

In [123]:
w_q = torch.nn.Linear(d_in, d_out)
w_k = torch.nn.Linear(d_in, d_out)
w_v = torch.nn.Linear(d_in, d_out)
q = w_q(inputs)
k = w_k(inputs)
v = w_v(inputs)
q2 = q[2]
q2

tensor([-0.0541, -0.5428], grad_fn=<SelectBackward0>)

In [124]:
attn_score = q2 @ v.T
attn_weights = torch.softmax(attn_score/d_out**0.5, dim=-1)
attn_weights

tensor([0.1854, 0.1601, 0.1611, 0.1724, 0.1562, 0.1648],
       grad_fn=<SoftmaxBackward0>)

In [125]:
c_2 = attn_weights @ v
c_2

tensor([-0.6476, -0.6053], grad_fn=<SqueezeBackward4>)

In [126]:
query = model.w_q(inputs)
keys = model.w_k(inputs)
values = model.w_v(inputs)
query, keys, values

(tensor([[ 0.3007,  0.0702],
         [ 0.3209,  0.1017],
         [ 0.3652, -0.1644],
         [ 0.2319,  0.1432],
         [ 0.3331,  0.0740],
         [ 0.3424,  0.1070]], grad_fn=<AddmmBackward0>),
 tensor([[ 0.8451,  0.2828],
         [ 0.5211, -0.0177],
         [ 0.5422,  0.3039],
         [ 0.5145,  0.6384],
         [ 0.4744, -0.0566],
         [ 0.6498, -0.1987]], grad_fn=<AddmmBackward0>),
 tensor([[0.1641, 0.2629],
         [0.3219, 0.5026],
         [0.1518, 0.3610],
         [0.2117, 0.0906],
         [0.3314, 0.5417],
         [0.3245, 0.5882]], grad_fn=<AddmmBackward0>))

In [ ]:
print(query.shapem,

In [127]:
attn_weights = query @ keys.T
attn_weights

tensor([[0.2740, 0.1555, 0.1844, 0.1995, 0.1387, 0.1815],
        [0.3000, 0.1654, 0.2049, 0.2300, 0.1465, 0.1883],
        [0.2621, 0.1932, 0.1480, 0.0829, 0.1826, 0.2700],
        [0.2364, 0.1183, 0.1692, 0.2107, 0.1019, 0.1222],
        [0.3025, 0.1723, 0.2031, 0.2187, 0.1539, 0.2018],
        [0.3196, 0.1765, 0.2181, 0.2444, 0.1564, 0.2012]],
       grad_fn=<MmBackward0>)

In [128]:
attn_scores = torch.softmax(attn_weights/d_out**0.5, dim=-1)
attn_scores

tensor([[0.1769, 0.1627, 0.1661, 0.1678, 0.1608, 0.1657],
        [0.1780, 0.1619, 0.1664, 0.1694, 0.1597, 0.1645],
        [0.1752, 0.1669, 0.1616, 0.1544, 0.1656, 0.1762],
        [0.1758, 0.1617, 0.1677, 0.1727, 0.1599, 0.1622],
        [0.1780, 0.1623, 0.1659, 0.1678, 0.1602, 0.1658],
        [0.1788, 0.1616, 0.1664, 0.1695, 0.1593, 0.1644]],
       grad_fn=<SoftmaxBackward0>)

In [129]:
context_vectors = attn_scores @ keys
context_vectors

tensor([[0.5946, 0.1628],
        [0.5949, 0.1645],
        [0.5952, 0.1499],
        [0.5939, 0.1668],
        [0.5950, 0.1630],
        [0.5952, 0.1648]], grad_fn=<MmBackward0>)

In [150]:
class Self_Attns(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.w_k = nn.Linear(d_in, d_out, qkv_bias)
        self.w_q = nn.Linear(d_in, d_out, qkv_bias)
        self.w_v = nn.Linear(d_in, d_out, qkv_bias)

    def forward(self,x):
        query = self.w_k(x)
        keys = self.w_q(x)
        values = self.w_v(x)

        attn_weights = query @ keys.T # 6,6
        attn_score = torch.softmax(attn_weights/d_k**0.5, dim=-1)

        context_vect = attn_score @ values 
        return context_vect


    

In [151]:
d_in = 3
d_out = 2

model = Self_Attns(d_in, d_out)
model(inputs)

tensor([[-0.3141,  0.0965],
        [-0.3209,  0.1012],
        [-0.3154,  0.0972],
        [-0.3277,  0.1055],
        [-0.3210,  0.1013],
        [-0.3162,  0.0981]], grad_fn=<MmBackward0>)

### Casual Attention 

In [156]:
mask = torch.tril(torch.ones(6, 6))

In [157]:
attn_weights * mask

tensor([[0.2740, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3000, 0.1654, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2621, 0.1932, 0.1480, 0.0000, 0.0000, 0.0000],
        [0.2364, 0.1183, 0.1692, 0.2107, 0.0000, 0.0000],
        [0.3025, 0.1723, 0.2031, 0.2187, 0.1539, 0.0000],
        [0.3196, 0.1765, 0.2181, 0.2444, 0.1564, 0.2012]],
       grad_fn=<MulBackward0>)

In [160]:
context_length = 6
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.1769,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1780, 0.1619,   -inf,   -inf,   -inf,   -inf],
        [0.1752, 0.1669, 0.1616,   -inf,   -inf,   -inf],
        [0.1758, 0.1617, 0.1677, 0.1727,   -inf,   -inf],
        [0.1780, 0.1623, 0.1659, 0.1678, 0.1602,   -inf],
        [0.1788, 0.1616, 0.1664, 0.1695, 0.1593, 0.1644]],
       grad_fn=<MaskedFillBackward0>)


In [161]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5029, 0.4971, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3351, 0.3331, 0.3319, 0.0000, 0.0000, 0.0000],
        [0.2511, 0.2486, 0.2497, 0.2506, 0.0000, 0.0000],
        [0.2016, 0.1994, 0.1999, 0.2001, 0.1991, 0.0000],
        [0.1681, 0.1661, 0.1666, 0.1670, 0.1658, 0.1664]],
       grad_fn=<SoftmaxBackward0>)


In [162]:
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
print(dropout(attn_weights))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0057, 0.9943, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6662, 0.6637, 0.0000, 0.0000, 0.0000],
        [0.5022, 0.0000, 0.4994, 0.5011, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4003, 0.3981, 0.0000],
        [0.3362, 0.3321, 0.0000, 0.0000, 0.0000, 0.3328]],
       grad_fn=<MulBackward0>)


In [163]:
batch = torch.stack((inputs, inputs))
batch

tensor([[[0.3190, 0.7017, 0.8702],
         [0.6653, 0.1729, 0.6069],
         [0.0530, 0.3658, 0.3833],
         [0.3003, 0.9855, 0.1892],
         [0.6645, 0.0772, 0.5603],
         [0.7505, 0.0374, 0.9053]],

        [[0.3190, 0.7017, 0.8702],
         [0.6653, 0.1729, 0.6069],
         [0.0530, 0.3658, 0.3833],
         [0.3003, 0.9855, 0.1892],
         [0.6645, 0.0772, 0.5603],
         [0.7505, 0.0374, 0.9053]]])

In [21]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method. 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec


In [24]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 
context_len = batch.shape[1]
d_in =3
d_out = 2
ca = CausalAttention(d_in, d_out, context_len, 0)
ca(batch)

torch.Size([2, 6, 3])


tensor([[[0.8090, 0.4026],
         [0.7070, 0.2527],
         [0.5922, 0.2403],
         [0.6147, 0.1907],
         [0.5999, 0.1657],
         [0.6173, 0.1775]],

        [[0.8090, 0.4026],
         [0.7070, 0.2527],
         [0.5922, 0.2403],
         [0.6147, 0.1907],
         [0.5999, 0.1657],
         [0.6173, 0.1775]]], grad_fn=<UnsafeViewBackward0>)

In [ ]:
class MultiHeadSelfAttention(nn.Moduel):
    def __init__(d_in, d_out, heads, context_len, dropout):
        super().__init__()
        
        assert (d_out/heads==0)
        
        self.d_out = d_out
        self.h = heads
        self.h_dim = d_out //heads
        
        self.dropout = dropout
        self.context_len = nn.Dropout(dropout) 
        
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False) 

        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)))
        
    def forwared(self, x):
        batch, num_tokens, dim = x
        h = self.heads
        h_dim = self.h_dim

        keys = self.W_k(x).view(b, num_tokens, h, h_dim)
        values = self.W_v(x).view(b, num_tokens, h, h_dim)
        query = self.W_q(x).view(b, num_tokens, h, h_dim)
        

        attn_weights = query @ keys.transpose(1,2)
        
        

        
        


In [48]:
import torch.nn as nn

heads = 2
d_in = 3
d_out = 4
context_len = 6
dropout = 0.5

h_dim = d_out//heads # 2 


dropout = nn.Dropout(dropout)
Wq = torch.nn.Linear(d_in, d_out, bias = False)
Wk = torch.nn.Linear(d_in, d_out, bias = False)
Wv = torch.nn.Linear(d_in, d_out, bias = False)
proj = torch.nn.Linear(d_out, d_out, bias = False)

b_n, no_tokens, dims = b.shape

querys = Wq(b)
keys = Wk(b)
values = Wv(b) # batches, no_tokens, (dims -> heads x h_dims)

querys = querys.reshape(b_n, no_tokens, heads, h_dim)
keys = keys.reshape(b_n, no_tokens, heads, h_dim)
values = values.reshape(b_n, no_tokens, heads, h_dim)

querys = querys.transpose(1,2)
keys = keys.transpose(1,2)
values = values.transpose(1,2)

attn_scores = querys @ keys.transpose(2,3)
mask = torch.triu(torch.ones(context_len, context_len,), diagonal=1)
attn_scores.masked_fill_(mask.bool(), -torch.inf)

attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
attn_weights = dropout(attn_weights)

# batches, no_tokes, heads, h_dims
context_vect = (attn_weights @ values).transpose(1,2)


# batches, no_tokes, d_out 
context_vect = context_vect.contiguous().view(b_n, no_tokens, d_out)
context_vect = proj(context_vect)

context_vect






tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.7582, -0.1108, -0.8121, -0.7845],
         [ 0.3757, -0.0020, -0.3939, -0.2927],
         [ 0.3831,  0.0216, -0.3890, -0.2546],
         [ 0.4130, -0.0170, -0.4360, -0.3682],
         [ 0.2904,  0.0523, -0.2726, -0.1592]],

        [[ 0.8237,  0.1906, -0.7619, -0.3003],
         [ 0.7201,  0.0373, -0.7362, -0.5124],
         [ 0.2832, -0.0648, -0.3147, -0.3307],
         [ 0.1785, -0.0569, -0.2178, -0.2486],
         [ 0.3709, -0.0474, -0.4015, -0.4090],
         [ 0.3422, -0.0058, -0.3623, -0.2792]]], grad_fn=<UnsafeViewBackward0>)

In [36]:
b = torch.stack((inputs,inputs))
batch, no_tokens, dim = b.shape
b

tensor([[[0.3190, 0.7017, 0.8702],
         [0.6653, 0.1729, 0.6069],
         [0.0530, 0.3658, 0.3833],
         [0.3003, 0.9855, 0.1892],
         [0.6645, 0.0772, 0.5603],
         [0.7505, 0.0374, 0.9053]],

        [[0.3190, 0.7017, 0.8702],
         [0.6653, 0.1729, 0.6069],
         [0.0530, 0.3658, 0.3833],
         [0.3003, 0.9855, 0.1892],
         [0.6645, 0.0772, 0.5603],
         [0.7505, 0.0374, 0.9053]]])

In [14]:
mask = torch.triu(torch.ones(context_len, context_len))
mask.masked_fill(mask.bool(), -torch.inf)

tensor([[-inf, -inf, -inf, -inf, -inf, -inf],
        [0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf]])

In [44]:
import torch.nn as nn

heads = 2
d_in = 3
d_out = 4
context_len = 6
dropout = 0.5

h_dim = d_out//heads # 2 


dropout = nn.Dropout(dropout)
Wq = torch.nn.Linear(d_in, d_out, bias = False)
Wk = torch.nn.Linear(d_in, d_out, bias = False)
Wv = torch.nn.Linear(d_in, d_out, bias = False)

b_n, no_tokens, dims = b.shape

querys = Wq(b)
keys = Wk(b)
values = Wv(b) # batches, no_tokens, (dims -> heads x h_dims)

querys = querys.reshape(b_n, no_tokens, heads, h_dim)
keys = keys.reshape(b_n, no_tokens, heads, h_dim)
values = values.reshape(b_n, no_tokens, heads, h_dim)

querys = querys.transpose(1,2)
keys = keys.transpose(1,2)
values = values.transpose(1,2)

attn_scores = querys @ keys.transpose(2,3)
mask = torch.triu(torch.ones(context_len, context_len))

attn_scores.masked_fill(mask.bool()[:][:], -torch.inf)
attn_scores

tensor([[[[ 0.3664,  0.1964,  0.1721,  0.2097,  0.1671,  0.2619],
          [ 0.1568,  0.0928,  0.0723,  0.0279,  0.0837,  0.1465],
          [ 0.1789,  0.0942,  0.0843,  0.1143,  0.0793,  0.1212],
          [ 0.3646,  0.1926,  0.1717,  0.2283,  0.1624,  0.2496],
          [ 0.1179,  0.0726,  0.0540,  0.0010,  0.0669,  0.1213],
          [ 0.1552,  0.0944,  0.0712,  0.0099,  0.0863,  0.1549]],

         [[ 0.1648,  0.1274,  0.0693,  0.1886,  0.1122,  0.1255],
          [ 0.0375,  0.0257,  0.0166,  0.0113,  0.0235,  0.0372],
          [ 0.0871,  0.0680,  0.0365,  0.1054,  0.0597,  0.0648],
          [ 0.1227,  0.1036,  0.0493,  0.2252,  0.0889,  0.0701],
          [ 0.0199,  0.0115,  0.0093, -0.0147,  0.0112,  0.0255],
          [ 0.0480,  0.0299,  0.0220, -0.0148,  0.0283,  0.0558]]],


        [[[ 0.3664,  0.1964,  0.1721,  0.2097,  0.1671,  0.2619],
          [ 0.1568,  0.0928,  0.0723,  0.0279,  0.0837,  0.1465],
          [ 0.1789,  0.0942,  0.0843,  0.1143,  0.0793,  0.1212],
    

tensor([[[ 0.0053,  0.3501, -0.0749, -0.4079],
         [ 0.0327,  0.3143, -0.0767, -0.4408],
         [ 0.0445,  0.2983, -0.0737, -0.4385],
         [ 0.0005,  0.2933, -0.0686, -0.3626],
         [ 0.0207,  0.2923, -0.0708, -0.3967],
         [ 0.0393,  0.2969, -0.0736, -0.4327]],

        [[ 0.0053,  0.3501, -0.0749, -0.4079],
         [ 0.0327,  0.3143, -0.0767, -0.4408],
         [ 0.0445,  0.2983, -0.0737, -0.4385],
         [ 0.0005,  0.2933, -0.0686, -0.3626],
         [ 0.0207,  0.2923, -0.0708, -0.3967],
         [ 0.0393,  0.2969, -0.0736, -0.4327]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
